[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhangdongming0607/ai-memory-course/blob/main/10-your-system.ipynb)

> 点击上方按钮，在 Google Colab 中打开本课，可以直接运行代码，无需本地环境。

# 第 10 课：回到你们的系统 — 读懂记忆系统架构

> 恭喜你走到最后一课！学完这节课，你会发现：你们系统里的每一块，你都已经学过了。

---

## 10.1 先回忆一下学了什么

过去 9 课，我们一步步拆解了 AI 记忆系统的核心概念。现在把它们和你们系统的模块对应起来：

| 课程 | 学到的概念 | 对应你们系统的模块 |
|------|----------|------------------|
| 第 1 课 LLM 基础 | Token、Prompt、Context Window、Temperature | 所有 LLM 调用的底层基础 |
| 第 2 课 调 LLM API | messages 格式、role 设定、API 调用流程 | keyboard_service_v4.py 发起的各种 LLM 请求 |
| 第 3 课 结构化提取 | 让 AI 输出 JSON、字段约束 | OCR 识别联系人 + 消息提取 |
| 第 4 课 数据库存储 | 事实数据持久化、表结构设计 | contact_conversation_facts 事实表 |
| 第 5 课 增量更新 | 旧摘要 + 新事实 → 新摘要 | memory layer 的更新策略 |
| 第 6 课 Prompt 注入 | 把记忆拼进 system message | memory_provider 组装上下文 |
| 第 7 课 Mem0 | 开源记忆框架、自动管理 | 对比理解你们系统为什么要自建 |
| 第 8 课 SSE | 流式输出、逐字返回 | chat stream 打字机效果 |
| 第 9 课 异步任务 | 后台队列、任务链 | Dramatiq worker 处理 ingest → persist → update |

看到没？**你已经具备了看懂整个系统的知识储备**。

这节课我们不写代码，就是把这些知识串起来，对照系统架构走一遍。走完之后，你再看后端代码，会有一种「哦原来就是这样」的感觉。

## 10.2 逐块解读系统架构

我们按照一次完整的用户操作流程来走：**用户上传截图 → 系统识别 → 存储 → 生成记忆 → 用记忆辅助回复**。

---

### 截图上传 → Usage 创建（第 2 课 API 调用 + 第 9 课异步任务）

用户在键盘上点「分析截图」，发生了什么？

1. **keyboard_service_v4.py** 收到图片 —— 就是一个普通的 API 接口，接收图片数据
2. 写入一条 **Usage 记录**，状态设为 `status=pending` —— 还记得第 9 课说的吗？异步任务的第一步就是「创建任务记录」
3. 图片缓存到 **Redis**，设置 600 秒过期 —— Redis 你可以理解为一个超快的临时存储，像是内存里的便利贴。600 秒后自动撕掉，因为图片处理完就不需要原图了
4. 计算 **image_fingerprint** 用于去重 —— 就是给图片算个「指纹」（hash 值），如果用户手抖传了两次一样的截图，第二次直接跳过

```
用户点击上传
    ↓
[keyboard_service_v4.py]
    ├── 创建 Usage 记录 (status=pending)
    ├── 图片 → Redis (TTL=600s)
    ├── 计算 image_fingerprint (去重)
    └── 发送异步任务 → Dramatiq 队列
```

到这里，API 就返回了。用户看到的是「正在分析中…」，后面的重活全交给后台 worker。

### 联系人识别 OCR/LLM（第 3 课结构化提取）

后台 worker 拿到任务后，第一步是**看懂截图里聊的是什么**。

这就是第 3 课学的结构化提取：

- 给 LLM 一张聊天截图
- 告诉它：「请提取联系人昵称、聊天平台、消息列表，以 JSON 输出」
- LLM 返回类似这样的结果：

```json
{
  "contact_name": "小明",
  "platform": "wechat",
  "messages": [
    {"sender": "contact", "content": "周末去爬山吗？"},
    {"sender": "user", "content": "好啊，去哪个山？"},
    {"sender": "contact", "content": "香山怎么样"}
  ]
}
```

**一个产品决策**：群聊直接跳过，不处理。为什么？

- 群聊里的人太多，谁说了什么跟「你和这个人的关系」关联不大
- 群聊消息量巨大，处理成本高
- 群聊场景下用户也不太需要「记住每个人说了啥」

这是个典型的「先做核心场景，复杂的后面再说」的决定。很合理。

### 事实层写入（第 4 课数据库存储）

识别出消息后，下一步是**存到数据库**。

这里存的是「原始事实」——谁在什么时候说了什么。对应的表是 `contact_conversation_facts`。

关键设计：**先删后写**。

```
同一个 usage_id 的事实
  ↓ 先删掉旧的（如果有的话）
  ↓ 再写入新的
```

为什么要这样？为了**幂等性**。

幂等性这个词听起来吓人，其实意思很简单：**同一个操作执行一次和执行十次，效果一样**。

想象一下：异步任务可能因为各种原因重试（网络抖动、worker 重启），如果不先删后写，重试一次就多一份重复数据。先删后写保证了「不管重试几次，结果都是对的」。

这其实是后端处理异步任务的常见模式，你以后看到类似设计就知道为什么了。

### 记忆层执行（第 3 课 + 第 5 课）

事实存好了，但事实不是记忆。你不会记住「小明 3 月 5 号下午说了'周末去爬山吗'」这种细节，你记住的是「小明最近想约我爬山」。

**记忆层就是把事实压缩成摘要的过程**。

它结合了两课的知识：
- **第 3 课**：让 AI 生成结构化摘要（输入事实，输出精炼的记忆文本）
- **第 5 课**：增量更新策略（不是每次从头生成，而是「旧摘要 + 新事实 → 新摘要」）

系统里有多个 **layer**（记忆层级），每个 layer 关注不同的维度：

| Layer | 关注什么 | 举例 |
|-------|---------|------|
| 关系小结 | 你和这个人的整体关系 | 「大学同学，毕业后同城，每周约饭」 |
| 社交 Tips | 跟这个人聊天要注意什么 | 「别提前女友，喜欢被夸」 |
| 近期动态 | 最近发生了什么 | 「在考虑跳槽，学吉他」 |

每个 layer 有自己的 **prompt 模板**和**更新策略**，都是通过配置决定的。

**关键点**：每个联系人的每个 layer 只保留**一条最新摘要**。不是追加，是覆盖。这样记忆永远保持精炼，不会越来越长把 context window 撑爆。

### 记忆注入回复（第 6 课 Prompt 注入）

终于到了用记忆的时候了。

用户要给小明回消息，系统怎么让 AI 「认识」小明？

1. **memory_provider** 从数据库读出小明的所有记忆层摘要
2. 把它们拼到 **system message** 里：

```python
system_prompt = f"""
你是用户的聊天助手，帮用户生成回复建议。

# 关于这个联系人
{relationship_summary}

# 聊天注意事项
{social_tips}

# 近期动态
{recent_updates}

请根据以上信息，生成自然得体的回复。
"""
```

就是第 6 课学的 Prompt 注入，没有任何黑魔法。

AI 之所以「知道」小明喜欢爬山、最近在学吉他，不是因为它真的认识小明，而是因为这些信息已经写在 prompt 里了。LLM 只是在「认真阅读说明书」然后给出合理的回复。

### 流式输出（第 8 课 SSE）

AI 生成回复的时候，用的是 **SSE（Server-Sent Events）** 流式返回。

```
服务器: data: {"content": "好"}
服务器: data: {"content": "的"}
服务器: data: {"content": "，"}
服务器: data: {"content": "周六"}
服务器: data: {"content": "早上"}
服务器: data: {"content": "出发"}
服务器: data: {"content": "！"}
```

前端拿到这些事件后逐字拼接渲染，就是你看到的「打字机效果」。

为什么不等 AI 全部生成完再一次返回？因为 LLM 生成一条回复可能要 2~5 秒，让用户干等体验很差。流式输出让用户从第一个字开始就能看到内容，感觉快多了。

### 异步任务链（第 9 课）

最后，把后台发生的事情串成一条链：

```
ingest_usage          →    persist_fact_memory     →    update_memory_layers
(OCR识别+提取消息)         (写入事实表)                  (更新各记忆层摘要)
```

这三步全部在 **Dramatiq worker** 中运行，跟 API 服务器是分开的进程。

为什么要用任务链而不是一个大函数搞定？

1. **可重试**：某一步失败了，只重试那一步，不用从头来
2. **可观测**：每一步都有状态记录，方便排查问题
3. **可扩展**：以后要加新的记忆层？只需要在最后一步加配置，前面完全不用动

这就是第 9 课讲的异步任务设计思想在实际系统中的应用。

## 10.3 架构全景图

把上面的所有环节串起来，就是你们记忆系统的完整流程：

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        用户操作                                        │
│   ┌──────────────┐              ┌──────────────┐                      │
│   │  上传聊天截图  │              │  发送消息请求  │                      │
│   └──────┬───────┘              └──────┬───────┘                      │
└──────────┼─────────────────────────────┼──────────────────────────────┘
           │                             │
           ▼                             ▼
┌────────────────────┐        ┌────────────────────────┐
│  API 层             │        │  Chat API              │
│  keyboard_service   │        │                        │
│  ·创建 Usage        │        │  ·memory_provider      │
│  ·图片→Redis(2,9课) │        │   读取记忆 ──(6课)──┐  │
│  ·image_fingerprint │        │  ·拼装 system prompt │  │
│  ·发送异步任务       │        │  ·调 LLM API ──(2课)│  │
└─────────┬──────────┘        │  ·SSE 流式返回(8课)  │  │
          │                    └────────────────────────┘
          ▼
┌─────────────────────────────────────────────┐
│  Dramatiq Worker（后台异步任务链）── (9课)    │
│                                             │
│  Step 1: ingest_usage                       │
│  ┌─────────────────────────────────┐        │
│  │ ·从 Redis 取图片                 │        │
│  │ ·调 LLM 做 OCR 识别 ──(3课)     │        │
│  │ ·输出: 联系人+消息 JSON          │        │
│  │ ·群聊跳过（产品决策）             │        │
│  └──────────────┬──────────────────┘        │
│                 ▼                            │
│  Step 2: persist_fact_memory                │
│  ┌─────────────────────────────────┐        │
│  │ ·先删后写（幂等）──(4课)         │        │
│  │ ·写入 contact_conversation_facts │        │
│  └──────────────┬──────────────────┘        │
│                 ▼                            │
│  Step 3: update_memory_layers               │
│  ┌─────────────────────────────────┐        │
│  │ ·读取 layer 配置                 │        │
│  │ ·旧摘要+新事实→新摘要 ──(3+5课)  │        │
│  │ ·每联系人每layer只保留一条        │        │
│  └─────────────────────────────────┘        │
│                                             │
└─────────────────────────────────────────────┘
          │
          ▼
┌─────────────────────────────────────────────┐
│  数据库 ──(4课)                              │
│  ┌───────────────────┐ ┌──────────────────┐ │
│  │ conversation_facts│ │ memory_summaries │ │
│  │ (原始事实)         │ │ (记忆摘要)       │ │
│  └───────────────────┘ └──────────────────┘ │
└─────────────────────────────────────────────┘
```

**左边**是截图处理链路（写入方向），**右边**是聊天回复链路（读取方向）。

它们通过数据库连接：左边往数据库写记忆，右边从数据库读记忆。两条路径互不阻塞。

## 10.4 前端需要关心的部分

作为前端开发者，整个系统你不需要全懂，但要分清楚「哪些是你直接打交道的」和「哪些是理解就好的」。

### 你直接打交道的

**1. SSE 流式输出（第 8 课）—— 渲染打字机效果**

这是你写代码最多的地方之一：
- 建立 SSE 连接
- 监听 `message` 事件，拿到每个 chunk
- 逐字拼接渲染到 UI 上
- 处理结束信号、错误重连

**2. API 调用 —— 上传截图、发送消息、获取联系人详情**

标准的 RESTful 接口调用：
- `POST /usage` —— 上传截图
- `POST /chat` —— 发送消息（返回 SSE 流）
- `GET /contacts/:id` —— 获取联系人信息和记忆

**3. 状态轮询 —— 截图分析进度、记忆更新状态**

因为截图分析是异步的，前端需要轮询或监听状态变化：
- 截图上传后，轮询 Usage 状态：`pending → processing → completed`
- 记忆更新后，刷新联系人详情页的记忆展示

### 你不需要直接关心但要理解的

**记忆提取和更新的后台流程**

你不需要知道 Dramatiq worker 怎么配置、prompt 模板怎么写、数据库表结构长什么样。但你要理解：

- 为什么上传截图后不能立即看到记忆更新（因为是异步处理的）
- 为什么有时候记忆内容变了但联系人页面没更新（可能需要主动刷新）
- 为什么同一张截图传两次不会重复处理（image_fingerprint 去重）

**为什么后端要这么设计**

理解了设计思路，你就能：
- 跟后端更有效地沟通
- 出 bug 时快速判断是前端问题还是后端问题
- 参与架构讨论时有自己的见解

## 10.5 如果你要和后端讨论简化方案

学完第 7 课 Mem0，你应该有个感觉：现在的系统比 Mem0 复杂不少。这些复杂度到底值不值得？

### 必要的复杂度（不建议砍）

| 设计 | 为什么必要 |
|------|----------|
| 异步任务链 | LLM 调用慢（3~10秒），不能让用户卡在那里等 |
| 先删后写的幂等设计 | 异步任务必然会有重试，不幂等就会数据错乱 |
| 事实层和记忆层分离 | 原始数据不能丢，摘要可以随时重新生成 |
| SSE 流式输出 | 用户体验的底线，等 5 秒才看到完整回复是不可接受的 |

### 可以探讨简化的方向

| 当前设计 | 简化思路 | 代价 |
|---------|---------|------|
| 多个 memory layer 各有独立 prompt | 合并为一个统一的摘要 prompt | 灵活性降低，但维护成本大幅下降 |
| 自定义的记忆更新策略 | 用 Mem0 这类框架托管 | 失去细粒度控制，但开发速度快 |
| 每次上传都算 image_fingerprint | 简单用 usage 状态判断 | 极端情况可能重复处理，但省了计算开销 |
| 复杂的 layer 配置系统 | 硬编码几个固定层 | 不能动态调整，但早期够用 |

### 我的建议

如果你们团队要讨论简化：

1. **先问「这个复杂度解决了什么真实问题」** —— 如果只是「以后可能用到」，那就先砍掉
2. **从 Mem0 这样的框架开始原型验证** —— 等确认了不够用再自建
3. **保留事实层** —— 这是你的原始数据，有了它，上面的记忆逻辑随时可以换
4. **异步和 SSE 不要动** —— 这是用户体验的基础设施，省不了

## 10.6 课程总结与下一步

### 恭喜你完成了全部 10 课！

从「LLM 是什么」到「看懂整个记忆系统架构」，你走过了一条完整的学习路径。现在你具备了：

- **理解 LLM 的基本概念**（Token、Prompt、Temperature）
- **会调 LLM API** 并理解 messages 格式
- **知道如何从非结构化数据中提取结构化信息**
- **理解记忆的存储和更新机制**
- **掌握 Prompt 注入和 RAG 的原理**
- **了解 Mem0 等开源框架的能力和局限**
- **能处理 SSE 流式输出**
- **理解异步任务链的设计思想**
- **能读懂你们系统的完整架构**

### 推荐阅读的代码文件（按顺序）

如果你想进一步深入，建议按这个顺序读源码：

1. **keyboard_service_v4.py** —— 入口，看请求怎么进来的
2. **OCR/识别相关的 prompt 模板** —— 看实际的结构化提取 prompt 长什么样
3. **事实写入的逻辑** —— 看先删后写怎么实现的
4. **memory layer 的配置和更新逻辑** —— 看增量更新的实际代码
5. **memory_provider** —— 看记忆怎么注入到 prompt 的
6. **chat stream 的实现** —— 看 SSE 在后端怎么发出来的
7. **Dramatiq 任务定义** —— 看异步任务链怎么串起来的

### 推荐继续深入的方向

| 方向 | 说明 | 适合谁 |
|------|------|-------|
| Prompt Engineering | 系统性学习 prompt 设计技巧 | 所有人 |
| RAG 进阶 | 向量数据库、embedding 模型选型、检索策略 | 想做全栈的同学 |
| LLM 应用框架 | LangChain、LlamaIndex 等 | 想快速出原型的同学 |
| Agent 开发 | 多步推理、工具调用、自主决策 | 对 AI 有热情的同学 |
| Fine-tuning | 微调模型以适配特定场景 | 想深入 AI 的同学 |

### 最后一句

AI 记忆系统听起来玄乎，拆开来看其实就是：**调 API → 提取信息 → 存起来 → 压缩一下 → 塞回 prompt**。

你已经完全理解了这条链路。

以后不管是优化现有系统、还是从零搭建新的 AI 功能，这些知识都能用上。加油！